# Визуализация результатов проекта HRNet-W18 / Jacquard V2

Ноутбук собирает все картинки, нужные для отчёта, презентации и защиты:

1. **Датасет Jacquard V2.** Исходные данные → resize до 384×384 → маски (binary / angle / multitask) → шаги аугментации.
2. **Кривые обучения** для каждой модели и сводное сравнение.
3. **Эволюция предсказаний по эпохам** (1, 5, 10, 15, 20, 25, 30) для каждой модели на 1–3 фиксированных сценах.
4. **Лучшая эпоха каждой модели** на 3–5 сценах + per-bin IoU bar-chart.
5. **Side-by-side сравнение всех моделей** на тестовых сценах Jacquard V2 и Cornell Grasp Dataset.
6. **Дополнительно**: IoU как функция угла, depth contribution heatmap, каталог худших предсказаний.

Все картинки сохраняются в `outputs/viz/<RUN_TAG>/` и одновременно отображаются inline.

## 0. Параметры запуска

Подставьте свои пути. Поддерживаются режимы `colab`, `local`, `kaggle` — отличаются только корнем (Drive, локальная FS, ввод Kaggle).

In [ ]:
import os, sys

# ----- Где запускаемся -----
ENV = "local"   # "colab" | "local" | "kaggle"

# ----- Корень репозитория -----
if ENV == "colab":
    REPO_ROOT = "/content/HRNet_Grasp_Semantic_Segmentation"
    if not os.path.isdir(REPO_ROOT):
        # Клонируем, если ещё нет
        !git clone https://github.com/chelovekrazumnii2-png/HRNet_Grasp_Semantic_Segmentation.git $REPO_ROOT
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
    # На Colab датасеты и чекпоинты обычно живут на смонтированном Drive.
    JACQUARD_ROOT = os.path.join(DRIVE_ROOT, "datasets/JacquardV2")
    SPLITS_PATH   = os.path.join(DRIVE_ROOT, "HRNet/splits/jacquard_v2.json")
    CORNELL_ROOT  = os.path.join(DRIVE_ROOT, "datasets/Cornel_grasp_dataset")
    RUNS_BASE     = os.path.join(DRIVE_ROOT, "HRNet/runs")
elif ENV == "local":
    # Ноутбук лежит в <repo>/notebooks/, корень репозитория — на уровень выше.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if not os.path.isdir(os.path.join(REPO_ROOT, "grasp_seg")):
        # Если запустили из другой папки — пропишите путь вручную.
        REPO_ROOT = r"D:\Libraries\Documents\VScode\HRNet_Grasp_Semantic_Sagmentation\HRNet_Grasp_Semantic_Segmentation"
    JACQUARD_ROOT = os.path.join(REPO_ROOT, "datasets", "JacquardV2")
    SPLITS_PATH   = os.path.join(REPO_ROOT, "train_results", "splits", "jacquard_v2.json")
    CORNELL_ROOT  = os.path.join(REPO_ROOT, "datasets", "Cornel_grasp_dataset")
    RUNS_BASE     = os.path.join(REPO_ROOT, "train_results")
elif ENV == "kaggle":
    REPO_ROOT = "/kaggle/working/HRNet_Grasp_Semantic_Segmentation"
    DRIVE_ROOT = "/kaggle/input"
    JACQUARD_ROOT = os.path.join(DRIVE_ROOT, "jacquard-v2")
    SPLITS_PATH   = os.path.join(REPO_ROOT, "splits/jacquard_v2.json")
    CORNELL_ROOT  = os.path.join(DRIVE_ROOT, "cornell-grasp")
    RUNS_BASE     = os.path.join(DRIVE_ROOT, "hrnet-runs")
else:
    raise ValueError(ENV)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ----- Чекпоинты (имя_модели → директория с epoch_*.pth, best.pth, metrics.csv, resolved_config.yaml) -----
# Подправьте имена справа под фактические названия папок у вас (например
# 'hrnet_w18_rgb_multitask', 'hrnet_w18_rgbd_multitask', 'hrnet_w18_rgbd_angle').
RUNS = {
    "multitask_rgb":  os.path.join(RUNS_BASE, "hrnet_w18_rgb_multitask"),
    "multitask_rgbd": os.path.join(RUNS_BASE, "hrnet_w18_rgbd_multitask"),
    "angle_rgbd":     os.path.join(RUNS_BASE, "hrnet_w18_rgbd_angle"),
}

# ----- Куда сохранять PNG -----
OUT_DIR = os.path.join(REPO_ROOT, "outputs/viz/report")
os.makedirs(OUT_DIR, exist_ok=True)

# ----- Параметры выборки -----
NUM_EVOLUTION_SCENES = 2
NUM_BEST_SCENES      = 4
NUM_COMPARE_SCENES   = 4
NUM_CORNELL_SCENES   = 4
EPOCH_STEP           = 5
MAX_IOU_SAMPLES      = 200
DPI                  = 140
SEED                 = 0

print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"JACQUARD_ROOT = {JACQUARD_ROOT}")
print(f"SPLITS_PATH   = {SPLITS_PATH}")
print(f"CORNELL_ROOT  = {CORNELL_ROOT}")
print(f"OUT_DIR       = {OUT_DIR}")
print(f"RUNS          = {RUNS}")

In [ ]:
# Установка недостающих зависимостей (быстро; основные уже стоят)
%pip install -q matplotlib pandas scipy seaborn opencv-python pillow scikit-image tifffile imagecodecs imageio pyyaml tqdm timm
# В Colab torch предустановлен; для local — pip install torch torchvision
import importlib, torch
print(f"torch {torch.__version__}, CUDA {'yes' if torch.cuda.is_available() else 'no'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import random, numpy as np, matplotlib.pyplot as plt
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"]  = 10

from grasp_seg.data.splits import load_split
from grasp_seg.data import cornell as cornell_data
from grasp_seg.viz import (dataset_viz, metrics_viz, epoch_evolution,
                            eval_viz, compare_viz, extra_viz,
                            inference, decoder)

split = load_split(SPLITS_PATH)
print(f"split: train={len(split.train)}, val={len(split.val)}, test={len(split.test)}")
test_files = list(split.test)

def save(fig, name, subdir):
    p = os.path.join(OUT_DIR, subdir, name)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    fig.savefig(p, dpi=DPI, bbox_inches="tight")
    print(f"saved → {p}")
    return fig

## 1. Датасет Jacquard V2

Здесь показываем «жизненный цикл» одного сэмпла перед тем, как он попадёт на вход модели.

In [ ]:
rng = random.Random(SEED)
sample = rng.choice(test_files)
print(f"Сцена для демонстрации: {sample}")

In [ ]:
# 1.1 Исходное изображение + GT-прямоугольники
save(dataset_viz.figure_raw_with_grasps(sample), "01_raw_with_grasps.png", "dataset")
plt.show()

In [ ]:
# 1.2 Resize 1024×1024 → 384×384 + растеризация маски
save(dataset_viz.figure_resize_pipeline(sample, image_size=384), "02_resize_pipeline.png", "dataset")
plt.show()

In [ ]:
# 1.3 Целевые маски, подаваемые на разные модели (binary / angle / multitask)
save(dataset_viz.figure_mask_modes(sample, image_size=384), "03_mask_modes.png", "dataset")
plt.show()

In [ ]:
# 1.4 Compact-polygon (length_scale=1/3) vs полный grasp-rectangle
save(dataset_viz.figure_compact_vs_full(sample, image_size=384), "04_compact_vs_full.png", "dataset")
plt.show()

In [ ]:
# 1.5 Шаги аугментации (флипы, повороты, scale+translate, color jitter, шум RGB, jitter+dropout глубины)
save(dataset_viz.figure_augmentation_steps(sample, image_size=384, seed=SEED),
     "05_augmentation_steps.png", "dataset")
plt.show()

In [ ]:
# 1.6 Cornell Grasp Dataset — исходный кадр с GT-прямоугольниками
if os.path.isdir(CORNELL_ROOT):
    ids = cornell_data.list_scenes(CORNELL_ROOT)
    if ids:
        cs = cornell_data.load_scene(CORNELL_ROOT, rng.choice(ids))
        save(dataset_viz.figure_cornell_raw(cs), "06_cornell_raw.png", "dataset")
        plt.show()
    else:
        print(f"Cornell-сцены не найдены под {CORNELL_ROOT}")
else:
    print(f"CORNELL_ROOT не существует: {CORNELL_ROOT}; пропускаем.")

## 2. Кривые обучения

Первый блок — отдельная панель по каждой модели; второй — сводное сравнение всех моделей в одной фигуре.

In [ ]:
csv_paths = {}
for name, run_dir in RUNS.items():
    csv = os.path.join(run_dir, "metrics.csv")
    if os.path.isfile(csv):
        csv_paths[name] = csv
        save(metrics_viz.figure_single_run(csv, title=name),
             f"{name}_curves.png", "training")
        plt.show()
    else:
        print(f"[skip] {name}: metrics.csv не найден ({csv})")

# Сводное сравнение
if len(csv_paths) >= 2:
    save(metrics_viz.figure_compare_runs(csv_paths), "compare_runs.png", "training")
    plt.show()

# Краткая сводка по best.pth
for name, csv in csv_paths.items():
    try:
        s = metrics_viz.best_epoch_summary(csv, target="val_miou_fg")
        print(f"{name:>20}: best epoch={s['best_epoch']}, val_miou_fg={s['best_value']:.4f}, last={s['last_value']:.4f}")
    except KeyError:
        pass

## 3. Эволюция предсказаний по эпохам

Для каждой модели берём чекпоинты `epoch_001`, `epoch_005`, `epoch_010`, `epoch_015`, `epoch_020`, `epoch_025`, `epoch_030` (последний — финальный), прогоняем их на 1–3 фиксированных сценах и строим сетку «сцена × эпоха».

In [ ]:
evo_scenes = rng.sample(test_files, min(NUM_EVOLUTION_SCENES, len(test_files)))
print(f"Сцены: {evo_scenes}")
for name, run_dir in RUNS.items():
    if not os.path.isdir(run_dir):
        print(f"[skip] {name}: {run_dir} не существует")
        continue
    try:
        fig = epoch_evolution.figure_epoch_evolution(
            run_dir, evo_scenes, step=EPOCH_STEP,
            title=f"Эволюция предсказаний по эпохам — {name}",
        )
        save(fig, f"{name}.png", "epoch_evolution")
        plt.show()
    except Exception as e:
        print(f"[warn] {name}: {e}")

## 4. Лучшая эпоха каждой модели

Загружаем `best.pth`, прогоняем на 3–5 тестовых сценах, рисуем вход / GT-маску / предсказание / GT-rect vs decoded-rect / карту ошибок (TP/FP/FN). Дополнительно — bar-chart IoU по угловым бинам (для всех моделей сразу).

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

runners = []
for name, run_dir in RUNS.items():
    if not os.path.isdir(run_dir):
        continue
    try:
        runners.append(inference.ModelRunner(run_dir, checkpoint="best",
                                              name=name, device=device))
        print(f"loaded {runners[-1].info.name}: ckpt={os.path.basename(runners[-1].info.checkpoint_path)}, "
              f"epoch={runners[-1].info.epoch}")
    except Exception as e:
        print(f"[warn] {name}: {e}")

In [ ]:
best_scenes = rng.sample(test_files, min(NUM_BEST_SCENES, len(test_files)))
for runner in runners:
    fig = eval_viz.figure_best_epoch_scenes(runner, best_scenes)
    save(fig, f"{runner.info.name}_scenes.png", "best_epoch")
    plt.show()

In [ ]:
# Per-bin IoU bar-chart (для angle и multitask моделей сразу)
compatible = [r for r in runners if r.info.mask_mode in ("angle", "multitask")]
if compatible:
    fig = eval_viz.figure_per_class_iou(compatible, test_files,
                                          max_samples=MAX_IOU_SAMPLES)
    save(fig, "per_class_iou.png", "best_epoch")
    plt.show()

## 5. Сравнение моделей side-by-side

Каждая строка — отдельная сцена; колонки: [GT, модель 1, модель 2, …].

In [ ]:
# 5.1 Jacquard V2 (test-сплит)
cmp_scenes = rng.sample(test_files, min(NUM_COMPARE_SCENES, len(test_files)))
fig = compare_viz.figure_compare_models_jacquard(runners, cmp_scenes)
save(fig, "jacquard_test.png", "compare")
plt.show()

In [ ]:
# 5.2 Cornell Grasp Dataset (cross-domain)
if os.path.isdir(CORNELL_ROOT):
    ids = cornell_data.list_scenes(CORNELL_ROOT)
    if ids:
        ids_sel = rng.sample(ids, min(NUM_CORNELL_SCENES, len(ids)))
        scenes = [cornell_data.load_scene(CORNELL_ROOT, i) for i in ids_sel]
        fig = compare_viz.figure_compare_models_cornell(runners, scenes)
        save(fig, "cornell_test.png", "compare")
        plt.show()
    else:
        print("Cornell сцены не найдены")
else:
    print(f"CORNELL_ROOT не существует: {CORNELL_ROOT}; пропускаем секцию.")

## 6. Дополнительная аналитика

- **IoU vs угол** — где модель работает лучше/хуже (например, вертикальные захваты против диагональных).
- **Depth contribution heatmap** — поэлементная разница `pos(RGB-D) − pos(RGB)`. Красные регионы = глубина помогла, синие = глубина «увела».
- **Каталог худших предсказаний** — 6–10 сцен с наименьшим foreground IoU и эвристическим объяснением (низкий контраст, мало пикселей объекта, дыры в глубине).

In [ ]:
# 6.1 IoU × angle для каждой модели
for runner in runners:
    fig = extra_viz.figure_iou_vs_angle(runner, test_files,
                                         max_samples=MAX_IOU_SAMPLES)
    save(fig, f"iou_vs_angle_{runner.info.name}.png", "extras")
    plt.show()

In [ ]:
# 6.2 Depth contribution: нужна одна RGB-only и одна RGB-D модель
rgb_runs  = [r for r in runners if r.info.input_mode == "rgb"]
rgbd_runs = [r for r in runners if r.info.input_mode == "rgbd"]
if rgb_runs and rgbd_runs:
    rgb_r = rgb_runs[0]
    rgbd_r = next((r for r in rgbd_runs if r.info.mask_mode == rgb_r.info.mask_mode),
                  rgbd_runs[0])
    dscenes = rng.sample(test_files, 3)
    fig = extra_viz.figure_depth_contribution(rgb_r, rgbd_r, dscenes)
    save(fig, "depth_contribution.png", "extras")
    plt.show()
else:
    print("Нужна одна RGB-only и одна RGB-D модель — пропускаем.")

In [ ]:
# 6.3 Каталог худших предсказаний
for runner in runners:
    fig = extra_viz.figure_failure_catalog(runner, test_files,
                                            n_show=8, max_samples=MAX_IOU_SAMPLES)
    save(fig, f"failures_{runner.info.name}.png", "extras")
    plt.show()

## 7. Сводная таблица top-1 grasp accuracy

Стандартная для Jacquard метрика: предсказание считается успешным, если `IoU > 0.25 ∧ |Δθ| < 30°` хотя бы с одним GT-захватом.

In [ ]:
import pandas as pd
from grasp_seg.viz.decoder import DecodeConfig

decode_cfg = DecodeConfig()
rows = []
eval_files = test_files[:MAX_IOU_SAMPLES]
for runner in runners:
    n_top1 = n_top5 = 0
    for gf in eval_files:
        rgb, depth, grasps, _ = dataset_viz.load_jacquard_scene(gf, image_size=runner.image_size)
        if not grasps:
            continue
        pred = runner.predict(rgb=rgb, depth=depth)
        if runner.info.mask_mode == "angle":
            decoded = decoder.decode_angle(pred["fg_conf"], pred["argmax"],
                                            runner.info.num_angle_bins, cfg=decode_cfg)
        elif runner.info.mask_mode == "multitask":
            decoded = decoder.decode_multitask(pred["pos"], pred["cos2t"],
                                                pred["sin2t"], pred["width"], cfg=decode_cfg)
        else:
            decoded = []
        if decoded:
            top1, _, _ = decoder.jacquard_match(decoded[0][0], grasps)
            n_top1 += int(top1)
            top5 = any(decoder.jacquard_match(g, grasps)[0] for g, _ in decoded[:5])
            n_top5 += int(top5)
    rows.append({
        "model": runner.info.name,
        "top-1 acc": n_top1 / max(len(eval_files), 1),
        "top-5 acc": n_top5 / max(len(eval_files), 1),
        "n_scenes":  len(eval_files),
    })
summary = pd.DataFrame(rows)
summary

## Готово
Все картинки лежат в `OUT_DIR`. Их можно вставлять в отчёт / презентацию напрямую.